# Ensamble analysis V1 pasive imaging 𓆝 𓆟 𓆞 𓆝 𓆟𓆝 𓆟 𓆞 𓆝 𓆟
---
This notebook shows the analysis pipeline for searching ensambles on two-photon calcium imaging data. This analysis is based on the results of analyses of individual neurons and pairs of neurons. These analyses revealed that BTBR exhibited reduced selectivity for orientation and direction, as well as greater noise correlation than the control group, raising questions regarding the level of population organization. This notebook assumes 2 experimental groups. 
The main question that guides this analysis is Can both the BTBR mice and C57B6 mice show ensamblatic activity on a pasive stimulation task? *Is equaly organized represented in both genotypes?*. 

## Imports and general setup

We first need to import the Python libraries we will use in the rest of the notebook.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt   
from scipy import stats
from sklearn.covariance import LedoitWolf
from pathlib import Path

<div style="background-color:##99f720; border-left: 6px solid #99f720; padding: 10px; border-radius: 4px;">
    <h4 style="margin-top: 0;">What happens if a folder name isn't recognized?</h4>
    If a group folder is named something like BTBR_V1_copy it won't either match "BTBR" or "C57". When that happens `detect_genotype` raises an error instead of skipping the folder silently. Not doing it wouldn't break anything immediately, but that animal would silently drop out of the analysis (you might report N=12 when only 11 were actually processed, and nobody would notice until the sample sizes stop adding up.). The tradeoff is that if the pipeline stops, someone has to check it by hand. 
</div>

In [6]:
# ₊˚ʚ ROUTES ⠂⠁⠈⠂⠄⠄⠂⠁⠁⠂⠄⠄⠂⠁⠁⠂⠂⠁⠈⠂⠄⠄⠂⠁⠁⠂⠄⠄⠂⠁⠁⠂⠂⠁⠈⠂⠄⠄⠂⠁⠁⠂⠄⠄⠂⠁⠁⠂
DATA_ROOT = r'D:/AL-205/Nicole/Calcio/Nueva_carpeta'  # <── change this

# Keywords for detecting files (case-insensitive)
SCOPE_KEYWORD  = 'reg'     # TXT scope data
STIM_KEYWORD   = 'pas'     # TXT stimuli
TRACES_KEYWORD = 'traces'  # CSV calcium traces

# group folders → genotype
GENOTYPE_FOLDERS = {
    'BTBR': 'BTBR',
    'C57':  'C57',
}

# ₊˚ʚ  AQUISICION ⠂⠁⠈⠂⠄⠄⠂⠁⠁⠂⠄⠄⠂⠁⠁⠂⠂⠁⠈⠂⠄⠄⠂⠁⠁⠂⠄⠄⠂⠁⠁⠂⠂⠁⠈⠂⠄⠄⠂⠁⠁⠂⠄⠄⠂⠁⠁
SAMPLING_RATE = 5           # Hz
MS_PER_FRAME  = 1000 / SAMPLING_RATE  # 200 ms nominal — measured ~190ms in raw hardware counter, see Section 0. Pending resolution

# ₊˚ʚ  STIM WINDOWS ⠂⠁⠈⠂⠄⠄⠂⠁⠁⠂⠄⠄⠂⠁⠁⠂⠂⠁⠈⠂⠄⠄⠂⠁⠁⠂⠄⠄⠂⠁⠁⠂⠂⠁⠈⠂⠄⠄⠂⠁⠁⠂⠄⠄⠂⠁
SEC_BEFORE    = 2   # baseline window, matches stimulus protocol (see Methods)
SEC_AFTER     = 4   # stimulus window, matches trial duration
FRAMES_BEFORE = int(SEC_BEFORE * SAMPLING_RATE)   # 10
FRAMES_AFTER  = int(SEC_AFTER  * SAMPLING_RATE)   # 20
TOTAL_FRAMES  = FRAMES_BEFORE + FRAMES_AFTER       # 30

# Response window for used to compute per-angle mean response (tuning curve ➝ OSI/DSI)
RESP_START = FRAMES_BEFORE      # frame 10
RESP_END   = TOTAL_FRAMES       # frame 30

# ₊˚ʚ  THRESHOLD ⠂⠁⠈⠂⠄⠄⠂⠁⠁⠂⠄⠄⠂⠁⠁⠂⠂⠁⠈⠂⠄⠄⠂⠁⠁⠂⠄⠄⠂⠁⠁⠂⠂⠁⠈⠂⠄⠄⠂⠁⠁⠂⠄⠄⠂⠁
RESPONSIVE_STD   = 3    # STDs from baseline for a responsive neuron TODO: cite source
OSI_THRESH       = 0.7  # threshold for orientation-selective neuron, per TODO: cite source

# ₊˚ʚ  OUTPUT ⠂⠁⠈⠂⠄⠄⠂⠁⠁⠂⠄⠄⠂⠁⠁⠂⠂⠁⠈⠂⠄⠄⠂⠁⠁⠂⠄⠄⠂⠁⠁⠂⠂⠁⠈⠂⠄⠄⠂⠁⠁⠂⠄⠄⠂⠁⠁⠂
OUTPUT_DIR = Path(DATA_ROOT) / 'pipeline_output_ensambles'
OUTPUT_DIR.mkdir(exist_ok=True)

print('Setup ready')
print(f'Output → {OUTPUT_DIR}')

Setup ready
Output → D:\AL-205\Nicole\Calcio\Nueva_carpeta\pipeline_output_ensambles


In [7]:
# FUNTION MODULE
def find_file( folder: Path, keyword: str, preferred_extension: str = None) -> Path:
    """
    Searches within `folder` for a single file whose name contains `keyword`
    (case-insensitive).

    If there are multiple candidates and `preferred_extension` is specified, the results are filtered
    by that extension before a decision is made. If, after filtering, there are still
    more than one, or if `preferred_extension` is not specified and there is more than one candidate,
    an error is raised: the ambiguity must be resolved manually; the system never simply selects
    “the first one that appears.”
    """
    kw = keyword.lower()
    matches= [f for f in folder.iterdir() if f.is_file() and kw in f.name.lower()]

    if not matches:
        raise FileNotFoundError(f"Could'nt find any file with {keyword} in {folder}.")

    if preferred_extension and len(matches) > 1:
        filtered = [f for f in matches if f.suffix.lower() == preferred_extension.lower()]
        if filtered:
            matches = filtered

    if len(matches) > 1:
        names = [f.name for f in matches]
        raise ValueError(
            f"Ambiguity when searching for ‘{keyword}’ in {folder}: {len(matches)} candidates {names}. "
            "Resolve this manually before continuing"
        )
    return matches[0]

def detect_genotype(group_name: str) -> list:
    for kw, geno in GENOTYPE_FOLDERS.items():
        if kw.upper() in group_name.upper():
             raise ValueError(
                f"the genotype '{group_name}' in the folder can't be recognized."
                f"Known folders: {list(GENOTYPE_FOLDES.keys())}"
            )

def discover_animals (root: Path, genotype: str) -> list:
    """
    Discover all animals in the root folder for a given genotype.
    """
    root= Path (root)
    animals= []
    for group in sorted(root.iterdir()):
        if not group.is_dir() or group.name.startswith(".") or group.name== "pipeline_output":
            continue
        genotype = detect_genotype(group.name)
        for animal in sorted(group.iterdir()):
            if not animal.is_dir() or animal.name.startswith("."):
                continue
            animals.append({
                'Id': animal.name,
                'genotype': genotype,
                'group': group.name,
                'folder': animal,
            })
    if not animals:
        raise ValueError(f"No animals found in {root} for genotype {genotype}.")
    return animals

#Find files
def find_file(folder: Path, keyword: str) -> Path:
    """Finds the first file whose name contains keyword (case-insensitive).
    Excludes files that also contain other keywords, to avoid the
    traces CSV being detected as the stimulus TXT."""
    kw = keyword.lower()
    matches = [f for f in folder.iterdir()
               if f.is_file() and kw in f.name.lower()]
    if not matches:
        raise FileNotFoundError(
            f"No file found with '{keyword}' in {folder}")
    # If there are several, prefer TXT over CSV for log/stimulus keywords
    txt_matches = [f for f in matches if f.suffix.lower() == '.txt']
    csv_matches = [f for f in matches if f.suffix.lower() == '.csv']
    if kw in ['reg', 'pas']:
        return txt_matches[0] if txt_matches else matches[0]
    else:  # traces → prefer CSV
        return csv_matches[0] if csv_matches else matches[0]


def detect_genotype(group_name: str) -> str:
    for kw, geno in GENOTYPE_FOLDERS.items():
        if kw.upper() in group_name.upper():
            return geno
    return 'Unknown'


def discover_subjects(root: str) -> list:
    """Walks two levels: group/subject."""
    root = Path(root)
    subjects = []
    for group in sorted(root.iterdir()):
        if not group.is_dir() or group.name.startswith('.') \
                or group.name == 'pipeline_output':
            continue
        genotype = detect_genotype(group.name)
        for subject in sorted(group.iterdir()):
            if not subject.is_dir() or subject.name.startswith('.'):
                continue
            subjects.append({
                'subject_id': subject.name,
                'genotype':   genotype,
                'group':      group.name,
                'folder':     subject,
            })
    return subjects

##read scope and stimuli

def parse_scope_txt(filepath: Path) -> tuple:
    """Parses the microscope TXT file.
    Format: HH:MM:SS.microsec.frameN.flag.channel
    Returns timestamp_strs, millisec (normalized to t=0), sensors."""
    with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
        lines = f.readlines()
    datasplit = [l.strip().split('.') for l in lines[1:]]
    datasplit = [d for d in datasplit if len(d) >= 4]
    timestamp_strs = [d[0] for d in datasplit]
    millisec  = np.array([float(d[2]) for d in datasplit])
    sensors   = np.array([float(d[3]) for d in datasplit])
    millisec  = millisec - millisec[0]
    return timestamp_strs, millisec, sensors


def parse_stim_txt(filepath: Path) -> tuple:
    """Parses the stimulus TXT file.
    Format: HH:MM:SS.mmm.FLAG
      11   → stimulus onset
      0XXX → angle when it ends (4s later)
    Returns tstamp_stlog, stimuli."""
    with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
        lines = f.readlines()
    datasplit    = [l.strip().split('.') for l in lines]
    datasplit    = [d for d in datasplit if len(d) >= 3]
    tstamp_stlog = [d[0] for d in datasplit]
    stimuli      = np.array([float(d[2]) for d in datasplit])
    return tstamp_stlog, stimuli


def get_clean_frame_times(millisec: np.ndarray, sensors: np.ndarray,
                           min_interval_ms: float = 100.0) -> np.ndarray:
    """Extracts real frame times (flag==23), removing duplicates."""
    framt    = np.where(sensors == 23)[0]
    t_fram   = millisec[framt]
    ll       = np.diff(t_fram)
    idx_keep = np.where(ll > min_interval_ms)[0] + 1
    return np.insert(t_fram[idx_keep], 0, t_fram[0])

## align stims and frames

def align_stimuli_to_frames(timestamp_strs, millisec, sensors,
                             tstamp_stlog, stimuli,
                             ms_per_frame: float = 200.0) -> np.ndarray:
    """Returns stimsnframes (n_angles, n_trials, 2):
       [:,:,0] = angle, [:,:,1] = onset frame.

    Note: n_trials is derived from angle counts only. The onset flag (11)
    is unreliable — it can register spuriously during acquisition — so its
    count is not used to determine trial number."""
    t_fram_clean = get_clean_frame_times(millisec, sensors)
    max_frame    = len(t_fram_clean) - 1

    stims_all, stim_counts = np.unique(stimuli, return_counts=True)
    stims    = stims_all[stims_all != 11]
    n_trials = int(np.min(stim_counts[stims_all != 11]))

    idx_t_stim = np.zeros((n_trials, len(stims)))

    ts_arr = np.array(timestamp_strs)
    for s, stim in enumerate(stims):
        t_stamp_idx = np.where(stimuli == stim)[0]
        t_stamp     = [tstamp_stlog[i - 1] for i in t_stamp_idx]
        for j, ts in enumerate(t_stamp):
            if j >= n_trials:
                break
            hms = list(map(int, ts.split(':')))
            hms[2] += 1
            ts2 = f"{hms[0]:02}:{hms[1]:02}:{hms[2]:02}"
            idx_bin = np.concatenate((
                np.where(ts_arr == ts)[0],
                np.where(ts_arr == ts2)[0]
            ))
            if len(idx_bin) > 0:
                idx_stim_j = np.where(
                    sensors[idx_bin[0]:idx_bin[-1]] == 4
                )[0] + idx_bin[0]
                if len(idx_stim_j) > 0:
                    idx_t_stim[j, s] = idx_stim_j[0]

    t_Stim_ms   = millisec[idx_t_stim.astype(int)]
    tf          = np.round(t_Stim_ms / ms_per_frame).astype(int)
    tf_clamped  = np.clip(tf, 0, max_frame)
    late        = t_Stim_ms - t_fram_clean[tf_clamped] > ms_per_frame
    tf[late]   += 1
    tf          = np.clip(tf, 0, max_frame).T  # (n_angles, n_trials)

    stims_tiled  = np.tile(stims[:, np.newaxis], (1, tf.shape[1]))
    stimsnframes = np.stack([stims_tiled, tf], axis=2)
    return stimsnframes

## extract responsive neurons

def extract_aligned_signals(traces_csv: Path, stimsnframes: np.ndarray,
                             frames_before: int = FRAMES_BEFORE,
                             frames_after:  int = FRAMES_AFTER) -> dict:
    """Returns extracted_data: {angle: array(N_neurons, T_frames, N_trials)}.
    Normalization: dF/F0 with baseline = pre-stimulus window."""
    # Handle separator and decimal format for files exported in Spanish locale
    df = pd.read_csv(traces_csv, decimal=',', sep=None, engine='python')
    if df.columns[0].lower() in ['unnamed: 0', 'index', 'time', 'frame']:
        df.drop(df.columns[0], axis=1, inplace=True)
    # Ensure all values are numeric
    df = df.apply(pd.to_numeric, errors='coerce').fillna(0)

    extracted_data = {}
    n_frames_total = len(df)
    total = frames_before + frames_after
    F_global_median = np.median(df.values, axis=0)

    for ai in range(stimsnframes.shape[0]):
        for ti in range(stimsnframes.shape[1]):
            angle = int(stimsnframes[ai, ti, 0])
            sf    = int(stimsnframes[ai, ti, 1])
            start = sf - frames_before
            end   = sf + frames_after

            if start >= 0 and end <= n_frames_total:
                F = df.iloc[start:end].values
            else:
                # Pad with global median
                F = np.zeros((total, df.shape[1]))
                s_clip = max(0, start)
                e_clip = min(n_frames_total, end)
                src    = df.iloc[s_clip:e_clip].values
                dst_s  = s_clip - start
                F[:, :]        = F_global_median
                F[dst_s:dst_s + len(src), :] = src

            F0     = np.median(F[:frames_before], axis=0)
            signal = (F - F0) / (F0 + 1e-10)

            extracted_data.setdefault(angle, []).append(signal)

    for angle in extracted_data:
        stacked = np.stack(extracted_data[angle], axis=-1)   # (T, N, trials)
        extracted_data[angle] = np.transpose(stacked, (1, 0, 2))  # (N, T, trials)

    return extracted_data


def find_responsive_neurons(extracted_data: dict,
                             threshold_std: float = RESPONSIVE_STD) -> list:
    """A neuron is responsive if its mean response to any angle
    exceeds threshold_std * std(baseline)."""
    n_neurons = next(iter(extracted_data.values())).shape[0]
    responsive = []
    for n in range(n_neurons):
        for angle, data in extracted_data.items():
            d         = data[n]             # (T, trials)
            threshold = np.std(d[:FRAMES_BEFORE]) * threshold_std
            response  = np.mean(d[RESP_START:RESP_END])
            if response > threshold:
                responsive.append(n)
                break
    return responsive

#Final dataset

def build_population_dataset_nc(subject_data: dict) -> dict:
    """Version for noise correlations. Preserves per-subject structure —
    correlations are only meaningful within a subject, where pairs of
    neurons share the exact same trials. No cross-subject concatenation
    or trial truncation."""
    pop_data = {}
    genotypes = set(d['genotype'] for d in subject_data.values())

    for geno in genotypes:
        subjects_geno = {sid: d for sid, d in subject_data.items()
                        if d['genotype'] == geno}

        all_angle_sets = [set(d['extracted_data'].keys())
                          for d in subjects_geno.values()]
        common_angles  = sorted(set.intersection(*all_angle_sets))

        resp_by_subject_angle = {}
        for sid, data in subjects_geno.items():
            ext  = data['extracted_data']
            resp = data['responsive']
            resp_by_subject_angle[sid] = {
                angle: ext[angle][resp, RESP_START:RESP_END, :]  # (N_i, T_resp, trials_i)
                for angle in common_angles
            }

        pop_data[geno] = {
            'angles':                common_angles,
            'subject_ids':           list(subjects_geno.keys()),
            'resp_by_subject_angle': resp_by_subject_angle,
        }

    return pop_data

In [8]:
subjects = discover_subjects(DATA_ROOT)
print(f'Subjects found: {len(subjects)}')
for s in subjects:
    print(f"  [{s['genotype']:6}]  {s['group']}/{s['subject_id']}")

Subjects found: 12
  [BTBR  ]  BTBR/BTBR_R2_060725
  [BTBR  ]  BTBR/BTBR_R2_270923_hembra_pasiva_Isaac
  [BTBR  ]  BTBR/BTBRL1030825_Reg_12_12_25
  [BTBR  ]  BTBR/BTBRR1_280825_Reg_15_12_25
  [BTBR  ]  BTBR/BTBRSM_270923_hembra_isaac
  [C57   ]  C57/C57_SM_Isaac
  [C57   ]  C57/C57L10300325_pasiva
  [C57   ]  C57/C57L2_Isaac
  [C57   ]  C57/C57R1230825_Reg_15_12_25
  [C57   ]  C57/C57R20010525_pasiva
  [C57   ]  C57/C57R3230825_Reg_15_12_25
  [C57   ]  C57/Delta 4-22 wt


In [9]:
all_results  = []
subject_data = {}
failed       = []

for subject in subjects:
    sid    = subject['subject_id']
    folder = subject['folder']
    geno   = subject['genotype']
    sep    = '=' * 60
    print(f'\n{sep}')
    print(f'  Processing: {sid}  [{geno}]')
    print(sep)
    try:
        scope_file  = find_file(folder, SCOPE_KEYWORD)
        stim_file   = find_file(folder, STIM_KEYWORD)
        traces_file = find_file(folder, TRACES_KEYWORD)
        print(f'  scope  : {scope_file.name}')
        print(f'  stim   : {stim_file.name}')
        print(f'  traces : {traces_file.name}')

        ts_strs, millisec, sensors = parse_scope_txt(scope_file)
        tstamp_stlog, stimuli      = parse_stim_txt(stim_file)
        flags, counts = np.unique(sensors, return_counts=True)
        print(f'  Scope flags : {dict(zip(flags.astype(int), counts))}')
        print(f'  Stim flags  : {dict(zip(*np.unique(stimuli, return_counts=True)))}')

        t_clean = get_clean_frame_times(millisec, sensors)
        print(f'  Clean frames: {len(t_clean)}')

        stimsnframes = align_stimuli_to_frames(
            ts_strs, millisec, sensors,
            tstamp_stlog, stimuli,
            ms_per_frame=MS_PER_FRAME
        )
        print(f'  stimsnframes: {stimsnframes.shape}  '
              f'(angles={stimsnframes.shape[0]}, trials={stimsnframes.shape[1]})')

        extracted_data = extract_aligned_signals(traces_file, stimsnframes)
        sample_angle   = next(iter(extracted_data))
        n_neurons      = extracted_data[sample_angle].shape[0]
        print(f'  Angles: {sorted(extracted_data.keys())}')
        print(f'  Shape per angle: {extracted_data[sample_angle].shape}  '
              f'(neurons, frames, trials)')

        responsive = find_responsive_neurons(extracted_data)
        print(f'  Responsive: {len(responsive)} / {n_neurons} '
              f'({100*len(responsive)/n_neurons:.1f}%)')

        subject_data[sid] = {
            'extracted_data': extracted_data,
            'responsive':     responsive,
            'genotype':       geno,
        }
        all_results.append(sid)

    except Exception:
        print(f'  ERROR in {sid}:')
        traceback.print_exc()
        failed.append(sid)

sep = '=' * 60
print(f'\n{sep}')
print(f'Successful: {len(all_results)} / {len(subjects)}')
if failed:
    print(f'Failed: {failed}')


  Processing: BTBR_R2_060725  [BTBR]
  scope  : BTBRmachor2060725_reg112539.txt
  stim   : BTBRmachor2060725_pas112548.txt
  traces : machoR2_060725_BTBR_pasiva_OryDir_CTraces.csv
  Scope flags : {np.int64(2): np.int64(998), np.int64(4): np.int64(309), np.int64(5): np.int64(329), np.int64(23): np.int64(36803)}
  Stim flags  : {np.float64(0.0): np.int64(10), np.float64(11.0): np.int64(90), np.float64(45.0): np.int64(10), np.float64(90.0): np.int64(10), np.float64(135.0): np.int64(10), np.float64(180.0): np.int64(10), np.float64(225.0): np.int64(10), np.float64(270.0): np.int64(10), np.float64(315.0): np.int64(10)}
  Clean frames: 6071
  stimsnframes: (8, 10, 2)  (angles=8, trials=10)
  Angles: [0, 45, 90, 135, 180, 225, 270, 315]
  Shape per angle: (187, 30, 10)  (neurons, frames, trials)
  Responsive: 183 / 187 (97.9%)

  Processing: BTBR_R2_270923_hembra_pasiva_Isaac  [BTBR]
  scope  : BTBR_hembra_R2_día2_144142reg.txt
  stim   : BTBR_R2_hembra_dia2_PASIVA_pas.txt
  traces : BTBR_HEM

In [10]:
# Population dataset
pop_data = build_population_dataset_nc(subject_data)

for geno, pdata in pop_data.items():
    sample_angle = pdata['angles'][0]
    subject_ids  = pdata['subject_ids']

    n_neurons_per_subject = {
        sid: pdata['resp_by_subject_angle'][sid][sample_angle].shape[0]
        for sid in subject_ids
    }
    total_neurons = sum(n_neurons_per_subject.values())

    print(f'\n{geno}: {total_neurons} total responsive neurons across '
          f'{len(subject_ids)} subjects')
    print(f'  Angles: {pdata["angles"]}')
    print(f'  Neurons per subject: {n_neurons_per_subject}')
    print(f'  Subjects included: {subject_ids}')


BTBR: 461 total responsive neurons across 5 subjects
  Angles: [0, 45, 90, 135, 180, 225, 270, 315]
  Neurons per subject: {'BTBR_R2_060725': 183, 'BTBR_R2_270923_hembra_pasiva_Isaac': 93, 'BTBRL1030825_Reg_12_12_25': 44, 'BTBRR1_280825_Reg_15_12_25': 122, 'BTBRSM_270923_hembra_isaac': 19}
  Subjects included: ['BTBR_R2_060725', 'BTBR_R2_270923_hembra_pasiva_Isaac', 'BTBRL1030825_Reg_12_12_25', 'BTBRR1_280825_Reg_15_12_25', 'BTBRSM_270923_hembra_isaac']

C57: 851 total responsive neurons across 7 subjects
  Angles: [0, 45, 90, 135, 180, 225, 270, 315]
  Neurons per subject: {'C57_SM_Isaac': 65, 'C57L10300325_pasiva': 72, 'C57L2_Isaac': 56, 'C57R1230825_Reg_15_12_25': 125, 'C57R20010525_pasiva': 142, 'C57R3230825_Reg_15_12_25': 164, 'Delta 4-22 wt': 227}
  Subjects included: ['C57_SM_Isaac', 'C57L10300325_pasiva', 'C57L2_Isaac', 'C57R1230825_Reg_15_12_25', 'C57R20010525_pasiva', 'C57R3230825_Reg_15_12_25', 'Delta 4-22 wt']
